# 05 — Tracker Hyperparameter Tuning: BoTSORT

Attempts a full hyperparameter grid search for BoTSORT over a larger subset of
validation videos, picking up architecture decisions from
`configs/kartector_botsort_arch.json` (written by notebook 04).

> **See the deprecation notice below before running.**

## ⚠️ Deprecated — Not Used in Final Project

Hyperparameter tuning was attempted here and in notebook 05b, but the swept
configurations performed **worse** at track association than the architecture
baselines from notebooks 04/04b.

**Decision:** Tracker hyperparameter tuning has been abandoned.
All downstream notebooks (06, 07, 08) use the baseline arch configs from
notebooks 04/04b directly.

These notebooks are retained for reference only.

## 0 — Configuration

In [1]:
import json as _json
from pathlib import Path
import json, tempfile, itertools, random
import cv2
import numpy as np

REPO_ROOT    = Path("/home1/hendersonj6179@cgu.edu")

# ── Load architecture decisions from notebook 04 ──────────────────────────
_ARCH_CFG = REPO_ROOT / 'configs/kartector_botsort_arch.json'
if _ARCH_CFG.exists():
    _arch = _json.loads(_ARCH_CFG.read_text())
    print(f'Loaded arch config from {_ARCH_CFG}:')
    print(_json.dumps(_arch, indent=2))
    CONF            = _arch.get('conf', 0.20)
    POST_PROCESS    = _arch.get('post_process', False)
    MAX_GAP         = _arch.get('max_gap', 45)
    MAX_DIST_PCT    = _arch.get('max_dist_pct', 35.0)
    REID_OPTION     = _arch.get('reid_option', 'none')   # 'A', 'B', or 'none'
    REID_B_PARAMS   = _arch.get('reid_b_params', {})     # boxmot params when REID_OPTION=='B'
    _TRACKER_YAML   = REPO_ROOT / _arch.get('tracker_yaml', 'configs/kartector_botsort_reentry.yaml')
else:
    print(f'WARNING: {_ARCH_CFG} not found — run 04_tracker_architecture_botsort.ipynb first.')
    print('Falling back to defaults.')
    CONF = 0.20; POST_PROCESS = False; MAX_GAP = 45; MAX_DIST_PCT = 35.0
    REID_OPTION = 'none'; REID_B_PARAMS = {}
    _TRACKER_YAML = REPO_ROOT / 'configs/kartector_botsort_reentry.yaml'

print(f'  reid_option={REID_OPTION}  reid_b_params={REID_B_PARAMS}')
print(f'  conf={CONF}  post_process={POST_PROCESS}  max_gap={MAX_GAP}  max_dist_pct={MAX_DIST_PCT}')

MOT_DIR      = REPO_ROOT / "data/mot_dataset"
MOT_LV_DIR   = REPO_ROOT / "data/mot_dataset_longvideo"
YOLO_RUNS    = (REPO_ROOT / "runs/yolo26").resolve()
TRACKER_RUNS = (REPO_ROOT / "runs/trackers").resolve()
TRACKER_RUNS.mkdir(parents=True, exist_ok=True)

CHUNKS_DIR = REPO_ROOT / "data/processed/labelstudiochunks"

# Detection thresholds
# CONF already loaded from arch config above
IOU  = 0.7

# Trackers to compare — both ship with Ultralytics, no extra install needed
# 05 tunes BoTSORT only — compare trackers in 05c_evaluate_compare
TRACKERS = ["botsort.yaml"]

# Fraction of test videos to use during HP search (1.0 = all, 0.3 = 30% fastest)
# Final evaluation (Section 3) always uses 100% of test videos.
HP_SEARCH_SUBSET = 0.05

# Hyperparameter grid searched on the test set
# ── Hyperparameter grid ────────────────────────────────────────────────────
# det_conf is fixed from arch config (set in Section 0 above).
# match_thresh and track_buffer were roughly identified in 04; here we do a
# finer sweep together with the secondary BoTSORT knobs.
# Post-process settings (MAX_GAP, MAX_DIST_PCT) come from arch config.
# ── Hyperparameter grid ────────────────────────────────────────────────────
# track_buffer and match_thresh are included here for a full fine-tune sweep.
# 04/04b only swept on a single video so wider search is warranted.
# Here we fine-tune the BoTSORT-specific secondary knobs with wider ranges.
HP_GRID = {
    # ── Re-entry / buffering ──────────────────────────────────────────────
    # track_buffer: frames to keep a lost track alive (re-entry bridging)
    "track_buffer":      [10, 20, 30, 45, 60, 90],
    # match_thresh: max association cost; lower allows bigger positional jumps
    "match_thresh":      [0.25, 0.50, 0.75, 0.90, 0.99],
    # ── Detection score thresholds ────────────────────────────────────────
    # track_high_thresh: min score to enter first-stage association
    #   wider range allows exploring both strict and permissive detection gates
    "track_high_thresh": [0.10, 0.20, 0.40, 0.60],
    # track_low_thresh: min score for second-stage (recovery) association
    #   must be < track_high_thresh; low values help recover occluded objects
    "track_low_thresh":  [0.01, 0.10,0.20, 0.30, 0.40],
    # new_track_thresh: min score to initialise a brand-new track
    #   higher values suppress ghost tracks; lower values catch faint detections
    "new_track_thresh":  [0.10, 0.20, 0.35, 0.55, 0.65],
    # ── ReID / spatial gating (BoTSORT only) ──────────────────────────────
    # proximity_thresh: min IoU for a track to be considered proximate for ReID
    "proximity_thresh":  [0.05, 0.2, 0.5, 0.9],
    # appearance_thresh: min cosine similarity to accept a ReID match
    #   lower = more aggressive linking; higher = fewer false merges
    "appearance_thresh": [0.10, 0.30, 0.50, 0.70],
}

# Max HP combinations to try per tracker (random subsample if grid is large)
HP_MAX_TRIALS = 100

# HP_SEARCH_METRIC: ACE (count accuracy) | MOTA | IDF1
# BIAS_PENALTY: weight on |mean per-class CE| to penalise systematic bias.
# Composite score (ACE mode) = ACE + BIAS_PENALTY * abs(mean_class_CE)
HP_SEARCH_METRIC = "ACE"
BIAS_PENALTY     = 1.0   # set to 0 to disable bias penalisation

with open(REPO_ROOT / "data/splits/splits.json") as f:
    splits = json.load(f)
CLASSES = splits["classes"]

# Final model weights produced by 02_train_yolo.ipynb Section 2
FINAL_WEIGHTS = YOLO_RUNS / "final" / "weights" / "best.pt"

print("Config OK")
print(f"  Trackers       : {TRACKERS}")
print(f"  Final weights  : {FINAL_WEIGHTS}")
print(f"  HP_MAX_TRIALS  : {HP_MAX_TRIALS}")
print(f"  Classes        : {CLASSES}")


Config OK
  Trackers       : ['bytetrack.yaml', 'botsort.yaml']
  Final weights  : /Users/joh11678/.ws/KARTector/runs/yolo11/final/weights/best.pt
  HP_MAX_TRIALS  : 8
  Classes        : ['Boost', 'Charge', 'Defense', 'Glide', 'HP', 'Offense', 'Top Speed', 'Turn', 'Weight']


## 1 — Helpers

Post-processing and metric utilities. These are imported from `helpers.py`.

In [ ]:
from helpers import (
    _iou, _cx, _cy,
    _cross_class_nms,
    _merge_track_rows,
    _count_tracks_per_class,
)

## 2 — Hyperparameter Search (test set, BoTSORT)

Samples up to `HP_MAX_TRIALS` random combinations from `HP_GRID`, runs inference
on test-set videos using the final YOLO model, and selects the config with the
highest MOTA.

> Lower `HP_MAX_TRIALS` in Section 0 to reduce runtime.

In [ ]:
import sys, warnings as _warnings
import yaml as _ty_hp

if not FINAL_WEIGHTS.exists():
    raise FileNotFoundError(
        f"Final weights not found: {FINAL_WEIGHTS}\n"
        "Run 02_train_yolo.ipynb — Section 2 (Final Model) first.")

test_chunk_names = splits["mot_test_chunks"]
all_test_videos = sorted([
    CHUNKS_DIR / fname
    for fname in test_chunk_names
    if (CHUNKS_DIR / fname).exists()
])
print(f"Total test videos: {len(all_test_videos)}")

# ── HP-search subset ──────────────────────────────────────────────────────
random.seed(42)
n_subset = max(1, int(len(all_test_videos) * HP_SEARCH_SUBSET))
search_videos = random.sample(all_test_videos, n_subset)
print(f"HP search subset : {len(search_videos)} / {len(all_test_videos)} "
      f"({HP_SEARCH_SUBSET*100:.0f}%)")

# ── Separate tracker and post-processing keys ─────────────────────────────
_TRACKER_KEYS = ["det_conf", "det_iou",
                 "track_high_thresh", "track_low_thresh", "new_track_thresh",
                 "track_buffer", "match_thresh",
                 "proximity_thresh", "appearance_thresh"]
_PP_KEYS      = ["pp_cc_iou", "pp_merge_time", "pp_merge_dist", "pp_merge_interp"]

tracker_grid = {k: HP_GRID[k] for k in _TRACKER_KEYS if k in HP_GRID}
pp_grid      = {k: HP_GRID[k] for k in _PP_KEYS      if k in HP_GRID}

tracker_combos = list(itertools.product(*tracker_grid.values()))
pp_combos      = list(itertools.product(*pp_grid.values()))
all_combos     = [dict(**dict(zip(tracker_grid.keys(), tc)),
                       **dict(zip(pp_grid.keys(),      pc)))
                  for tc, pc in itertools.product(tracker_combos, pp_combos)]

random.shuffle(all_combos)
combos = all_combos[:HP_MAX_TRIALS]
print(f"HP trials per tracker: {len(combos)} / {len(all_combos)} total")

# ── Helper: load arch-yaml defaults as a flat HP dict ─────────────────────
def _load_arch_defaults():
    """Return HP dict from arch tracker yaml (the reentry baseline)."""
    defaults = {}
    if _arch and 'tracker_yaml' in _arch:
        _p = REPO_ROOT / _arch['tracker_yaml']
        if _p.exists():
            defaults = _ty_hp.safe_load(_p.read_text()) or {}
    return defaults

_arch_defaults = _load_arch_defaults()
print(f"Baseline (arch yaml) defaults: {_arch_defaults}")

# ── Helper: build _trial_hp from combo + arch defaults ────────────────────
def _build_trial_hp(tracker_hp):
    hp = dict(tracker_hp)
    hp.setdefault('track_buffer', _arch_defaults.get('track_buffer', 30))
    hp.setdefault('match_thresh', _arch_defaults.get('match_thresh', 0.70))
    if REID_OPTION != 'B':
        with_reid = _arch_defaults.get('with_reid', False)
        hp['with_reid'] = with_reid
        if with_reid:
            hp.setdefault('model', _arch_defaults.get('model', 'auto'))
    return hp

# ── Helper: run one config and return metric row dict ─────────────────────
def _run_and_score(label, tracker, hp_overrides, pp_hp, pred_dir, videos):
    """Run tracker, compute all metrics, return a flat dict row."""
    run_tracker_hp(FINAL_WEIGHTS, videos, pred_dir, tracker,
                   hp_overrides=hp_overrides, conf=CONF, pp_overrides=pp_hp)
    m = compute_mot_metrics(MOT_DIR / "test", pred_dir)
    df_cnt = compute_count_metrics(MOT_DIR / "test", pred_dir, CLASSES)
    per_cls = df_cnt[df_cnt["class"] != "MACRO_AVG"]
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore", RuntimeWarning)
        ace_val  = float(per_cls["ACE"].mean()) if not per_cls.empty else None
        bias_val = float(per_cls["CE"].mean())  if not per_cls.empty else None
    composite = ((ace_val + BIAS_PENALTY * abs(bias_val))
                 if (ace_val is not None and bias_val is not None) else None)
    return {
        "config":    label,
        "tracker":   tracker_name(tracker),
        "MOTA":      m.get("MOTA"),
        "IDF1":      m.get("IDF1"),
        "ID_Sw":     m.get("ID_Sw"),
        "ACE":       ace_val,
        "CE_bias":   bias_val,
        "composite": composite,
        "pred_dir":  str(pred_dir),
    }

# ── Results DataFrame ─────────────────────────────────────────────────────
# One row per trial: tracker + all HP params + all metrics.
# Best config = sort df_hp by chosen metric and take first row.
hp_rows = []

for tracker in TRACKERS:
    tname = tracker_name(tracker)
    print(f"\n{'='*60}\nHP search — {tname}  ({len(search_videos)} videos)  "
          f"[optimising {HP_SEARCH_METRIC}]\n{'='*60}", flush=True)

    # ── Baseline: run arch-yaml defaults ──────────────────────────────────
    print("  → [baseline] arch yaml defaults", flush=True)
    try:
        _baseline_pp = {"pp_merge_time": MAX_GAP, "pp_merge_dist": MAX_DIST_PCT,
                        "pp_merge_interp": True, "pp_cc_iou": None} if POST_PROCESS else {}
        _baseline_hp = _build_trial_hp({})
        _baseline_row = _run_and_score(
            "baseline", tracker, _baseline_hp, _baseline_pp,
            TRACKER_RUNS / tname / "baseline", search_videos)
        _baseline_row.update({"trial": -1, **{k: _arch_defaults.get(k) for k in tracker_grid}})
        hp_rows.append(_baseline_row)
        def _fmt(v, fmt=".4f"): return format(v, fmt) if v is not None else "None"
        print(f"      MOTA={_fmt(_baseline_row['MOTA'])} IDF1={_fmt(_baseline_row['IDF1'])} "
              f"ID_Sw={_baseline_row['ID_Sw']} ACE={_fmt(_baseline_row['ACE'])} "
              f"CE_bias={_fmt(_baseline_row['CE_bias'])} composite={_fmt(_baseline_row['composite'])}",
              flush=True)
    except Exception as _e:
        print(f"  [ERROR baseline] {_e}", flush=True)

    # ── HP sweep trials ───────────────────────────────────────────────────
    for trial_idx, combo in enumerate(combos):
        tracker_hp = {k: v for k, v in combo.items() if k in _TRACKER_KEYS}
        pp_hp      = {k: v for k, v in combo.items() if k in _PP_KEYS}
        if POST_PROCESS:
            pp_hp = {"pp_merge_time": MAX_GAP, "pp_merge_dist": MAX_DIST_PCT,
                     "pp_merge_interp": True, "pp_cc_iou": None}
        _trial_hp = _build_trial_hp(tracker_hp)
        pred_dir  = TRACKER_RUNS / tname / "hp_search" / f"trial_{trial_idx:03d}"
        print(f"  → [{trial_idx+1:3d}/{len(combos)}] {combo}", flush=True)
        try:
            row = _run_and_score("hp_search", tracker, _trial_hp, pp_hp, pred_dir, search_videos)
            row.update({"trial": trial_idx, **combo})
            hp_rows.append(row)
            def _fmt(v, fmt=".4f"): return format(v, fmt) if v is not None else "None"
            print(f"      MOTA={_fmt(row['MOTA'])} IDF1={_fmt(row['IDF1'])} "
                  f"ID_Sw={row['ID_Sw']} ACE={_fmt(row['ACE'])} "
                  f"CE_bias={_fmt(row['CE_bias'])} composite={_fmt(row['composite'])}",
                  flush=True)
        except Exception as _trial_err:
            print(f"  [ERROR trial {trial_idx}] {_trial_err}", flush=True)
            continue

# ── Build results DataFrame ───────────────────────────────────────────────
df_hp = pd.DataFrame(hp_rows)
df_hp.to_csv(TRACKER_RUNS / "hp_search_log.csv", index=False)
print(f"\nSaved {len(df_hp)} trial results → {TRACKER_RUNS}/hp_search_log.csv")

# ── Pick best config per tracker by HP_SEARCH_METRIC ─────────────────────
_METRIC_ASCENDING = {"MOTA": False, "IDF1": False, "ID_Sw": True,
                     "ACE": True, "composite": True, "CE_bias": True}
_sort_col  = HP_SEARCH_METRIC
_ascending = _METRIC_ASCENDING.get(_sort_col, False)

best_configs = {}   # tracker yaml -> best combo dict (HP params only)
best_metrics = {}   # tracker yaml -> best metrics dict

print(f"\nBest configs (sorted by {_sort_col}, ascending={_ascending}):")
for tracker in TRACKERS:
    tname = tracker_name(tracker)
    sub = df_hp[(df_hp["tracker"] == tname) & (df_hp["config"] == "hp_search")].dropna(subset=[_sort_col])
    if sub.empty:
        print(f"  {tname}: no valid trials")
        continue
    best_row = sub.sort_values(_sort_col, ascending=_ascending).iloc[0]
    best_configs[tracker] = {k: best_row[k].item() if hasattr(best_row[k], 'item') else best_row[k]
                              for k in combo if k in best_row.index}
    best_metrics[tracker] = {k: best_row[k]
                              for k in ["MOTA","IDF1","ID_Sw","ACE","CE_bias","composite"]}
    print(f"  {tname}:")
    print(f"    HP     : { {k:v for k,v in best_configs[tracker].items() if k in _TRACKER_KEYS} }")
    print(f"    Metrics: {best_metrics[tracker]}")

print("\ndf_hp preview (hp_search trials only):")
display(df_hp[df_hp["config"]=="hp_search"].sort_values(["tracker", _sort_col],
        ascending=[True, _ascending]).head(20))


## 3 — Final Evaluation with Best Hyperparameters

In [ ]:
# ── Final evaluation: baseline vs best-HP vs GT upper-bound ─────────────
import yaml as _ty2

# Use full test set for final evaluation
eval_videos = all_test_videos
print(f"Final eval on {len(eval_videos)} test videos")

# ── Helper: compute GT upper-bound count metrics ───────────────────────────
def _gt_count_metrics(gt_dir, classes):
    """Compute count metrics treating GT as both pred and gt (perfect counts)."""
    import os as _os
    n = len(classes)
    gt_dir = Path(gt_dir)
    rows = []
    for seq in sorted(_os.listdir(gt_dir)):
        gt_file = gt_dir / seq / "gt" / "gt.txt"
        if not gt_file.exists(): continue
        with _warnings.catch_warnings():
            _warnings.simplefilter("ignore")
            gt_data = np.loadtxt(gt_file, delimiter=",")
        if gt_data.ndim == 1: gt_data = gt_data.reshape(1, -1)
        if gt_data.shape[0] == 0 or gt_data.shape[1] < 8: continue
        gt_c = np.zeros(n, dtype=int)
        for i in range(n):
            mask = gt_data[:, 7].astype(int) == (i + 1)
            gt_c[i] = len(set(gt_data[mask, 1].astype(int))) if mask.any() else 0
        rows.append(gt_c)
    if not rows: return None
    arr = np.array(rows, dtype=float)
    # Perfect prediction: pred == gt, so ACE=0, CE=0, CA=1
    result = []
    for i, cls in enumerate(classes):
        result.append({"class": cls, "GT_mean": round(arr[:,i].mean(),2),
                       "Pred_mean": round(arr[:,i].mean(),2),
                       "CE": 0.0, "ACE": 0.0, "CA": 1.0})
    result.append({"class": "MACRO_AVG",
                   "GT_mean": round(arr.mean(),2), "Pred_mean": round(arr.mean(),2),
                   "CE": 0.0, "ACE": 0.0, "CA": 1.0})
    return pd.DataFrame(result)

# ── Run baseline and best-HP configs ──────────────────────────────────────
final_results = {}   # key: config label -> {metrics, hp, pred_dir, count_df}

for tracker in TRACKERS:
    tname = tracker_name(tracker)
    _pp_eval = ({"pp_merge_time": MAX_GAP, "pp_merge_dist": MAX_DIST_PCT,
                 "pp_merge_interp": True, "pp_cc_iou": None}
                if POST_PROCESS else {})

    configs_to_run = {
        "baseline": _build_trial_hp({}),
        "best_hp":  _build_trial_hp({k: v for k, v in best_configs.get(tracker, {}).items()
                                      if k in _TRACKER_KEYS}),
    }

    for cfg_label, hp in configs_to_run.items():
        pred_dir = TRACKER_RUNS / tname / f"final_{cfg_label}"
        print(f"\nRunning {tname} [{cfg_label}] on {len(eval_videos)} videos...", flush=True)
        print(f"  HP: { {k:v for k,v in hp.items() if k in _TRACKER_KEYS} }")
        try:
            run_tracker_hp(FINAL_WEIGHTS, eval_videos, pred_dir, tracker,
                           hp_overrides=hp, conf=CONF, pp_overrides=_pp_eval)
            m  = compute_mot_metrics(MOT_DIR / "test", pred_dir)
            df_cnt = compute_count_metrics(MOT_DIR / "test", pred_dir, CLASSES)
            per_cls = df_cnt[df_cnt["class"] != "MACRO_AVG"]
            with _warnings.catch_warnings():
                _warnings.simplefilter("ignore", RuntimeWarning)
                ace_val  = float(per_cls["ACE"].mean()) if not per_cls.empty else None
                bias_val = float(per_cls["CE"].mean())  if not per_cls.empty else None
            final_results[cfg_label] = {
                "metrics":  {**m, "ACE": ace_val, "CE_bias": bias_val},
                "hp":       {k: v.item() if hasattr(v,'item') else v for k,v in hp.items()},
                "pred_dir": str(pred_dir),
                "count_df": df_cnt,
            }
            print(f"  MOTA={m['MOTA']}  IDF1={m['IDF1']}  ID_Sw={m['ID_Sw']}  "
                  f"ACE={ace_val}  CE_bias={bias_val}", flush=True)
        except Exception as _e:
            print(f"  [ERROR {cfg_label}] {_e}", flush=True)

# ── GT upper bound ────────────────────────────────────────────────────────
_gt_cnt = _gt_count_metrics(MOT_DIR / "test", CLASSES)
final_results["gt_upper_bound"] = {
    "metrics":  {"MOTA": 1.0, "IDF1": 1.0, "ID_Sw": 0, "ACE": 0.0, "CE_bias": 0.0},
    "hp":       {},
    "pred_dir": str(MOT_DIR / "test"),
    "count_df": _gt_cnt,
}
print("\nGT upper bound added.")

# ── Summary table ─────────────────────────────────────────────────────────
df_final = pd.DataFrame([
    {"config": k, **v["metrics"]}
    for k, v in final_results.items()
])
print("\n── Final test-set results ──")
display(df_final.round(4))
df_final.to_csv(TRACKER_RUNS / "test_metrics.csv", index=False)
with open(TRACKER_RUNS / "test_metrics.json", "w") as f:
    json.dump({k: v["metrics"] for k, v in final_results.items()}, f, indent=2)
with open(TRACKER_RUNS / "best_configs.json", "w") as f:
    json.dump({k: v["hp"] for k, v in final_results.items()}, f, indent=2)
print("Saved test_metrics.csv / .json / best_configs.json")


## 3b — Per-Class Instance Count Metrics

Counts unique track IDs per class in predictions vs ground truth, averaged across
test videos.

Metrics per class:
- **CE** (Count Error): mean(pred − gt) — signed
- **ACE** (Absolute Count Error): mean |pred − gt|
- **CA** (Count Accuracy): mean min(pred,gt)/max(pred,gt) ∈ [0,1]

In [ ]:
# ── Per-class count metrics: baseline vs best-HP vs GT upper-bound ───────
_CONFIG_LABELS  = ["baseline", "best_hp", "gt_upper_bound"]
_CONFIG_COLORS  = ["#4C72B0", "#DD8452", "#2ca02c"]
_CONFIG_DISPLAY = {"baseline": "Baseline (arch)", "best_hp": "Best HP",
                    "gt_upper_bound": "GT upper bound"}

for cfg_label in _CONFIG_LABELS:
    if cfg_label not in final_results: continue
    df_cnt = final_results[cfg_label]["count_df"]
    if df_cnt is None: continue
    print(f"\n── {_CONFIG_DISPLAY[cfg_label]} — Per-Class Count Metrics ──")
    print(df_cnt.to_string(index=False))

# ── Bar chart: CE / ACE / CA per class, one group per config ──────────────
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
fig.suptitle("Per-Class Count Metrics — Test Set (Full)", fontweight="bold")

active_cfgs = [c for c in _CONFIG_LABELS
               if c in final_results and final_results[c]["count_df"] is not None]
n_cfgs = len(active_cfgs)
width  = 0.8 / n_cfgs

for ax, metric, ylabel in zip(axes,
    ["CE", "ACE", "CA"],
    ["Count Error (CE)", "Abs Count Error (ACE)", "Count Accuracy (CA)"]):
    x = np.arange(len(CLASSES))
    for ci, cfg_label in enumerate(active_cfgs):
        df = final_results[cfg_label]["count_df"]
        vals = [df.loc[df["class"] == c, metric].values[0]
                if c in df["class"].values else 0
                for c in CLASSES]
        offset = (ci - n_cfgs/2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=_CONFIG_DISPLAY[cfg_label],
               color=_CONFIG_COLORS[ci], alpha=0.85)
    if metric == "CE":
        ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_xticks(x); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
    ax.set_ylabel(ylabel); ax.set_title(ylabel)
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
out_path = TRACKER_RUNS / "count_metrics_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {out_path}")


## 4 — Metric Visualizations

In [ ]:
import seaborn as sns

# ── Bar chart: MOTA / IDF1 / ID_Sw for baseline vs best-HP vs GT ─────────
_CONFIG_LABELS  = ["baseline", "best_hp", "gt_upper_bound"]
_CONFIG_COLORS  = ["#4C72B0", "#DD8452", "#2ca02c"]
_CONFIG_DISPLAY = {"baseline": "Baseline (arch)", "best_hp": "Best HP",
                    "gt_upper_bound": "GT upper bound"}

active_cfgs = [c for c in _CONFIG_LABELS if c in final_results]
cfg_labels_disp = [_CONFIG_DISPLAY[c] for c in active_cfgs]
cfg_colors      = [_CONFIG_COLORS[i] for i, c in enumerate(_CONFIG_LABELS) if c in active_cfgs]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, metric in zip(axes, ["MOTA", "IDF1", "ID_Sw"]):
    vals = [final_results[c]["metrics"].get(metric) or 0 for c in active_cfgs]
    bars = ax.bar(cfg_labels_disp, vals, color=cfg_colors)
    ax.set_title(metric)
    ax.set_ylim(0, max(vals) * 1.3 if max(vals) > 0 else 1)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}" if metric != "ID_Sw" else str(int(val)),
                ha="center", va="bottom", fontsize=10)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Baseline vs Best HP vs GT Upper Bound — Test Set", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(TRACKER_RUNS / "metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {TRACKER_RUNS}/metric_comparison.png")

# ── ACE / CE_bias comparison ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric, ylabel in zip(axes,
    ["ACE", "CE_bias"],
    ["Abs Count Error (ACE)", "Count Error Bias (CE_bias)"]):
    vals = [final_results[c]["metrics"].get(metric) or 0 for c in active_cfgs]
    bars = ax.bar(cfg_labels_disp, vals, color=cfg_colors)
    if metric == "CE_bias":
        ax.axhline(0, color="black", lw=0.8, ls="--")
    ax.set_title(ylabel); ax.set_ylabel(ylabel)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=10)
    ax.tick_params(axis="x", rotation=15)
plt.suptitle("Count Accuracy Metrics — Test Set", y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig(TRACKER_RUNS / "count_accuracy_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {TRACKER_RUNS}/count_accuracy_comparison.png")

# ── HP search heat-map (HP_SEARCH_METRIC vs two key params) ──────────────
if "track_high_thresh" in HP_GRID and "match_thresh" in HP_GRID and not df_hp.empty:
    _hp_only = df_hp[df_hp["config"] == "hp_search"].copy()
    if not _hp_only.empty and not _hp_only[HP_SEARCH_METRIC].isna().all():
        fig, axes = plt.subplots(1, len(TRACKERS), figsize=(7*len(TRACKERS), 5))
        if len(TRACKERS) == 1: axes = [axes]
        for ax, tracker in zip(axes, TRACKERS):
            name = tracker_name(tracker)
            sub = _hp_only[_hp_only["tracker"] == name].copy()
            if sub.empty: ax.set_title(f"{name} — no data"); continue
            pivot = sub.pivot_table(index="track_high_thresh", columns="match_thresh",
                                    values=HP_SEARCH_METRIC, aggfunc="mean")
            sns.heatmap(pivot, annot=True, fmt=".3f", cmap="YlGnBu", ax=ax,
                        linewidths=0.5, cbar_kws={"label": HP_SEARCH_METRIC})
            ax.set_title(f"{name} — {HP_SEARCH_METRIC} (track_high_thresh × match_thresh)")
        plt.tight_layout()
        plt.savefig(TRACKER_RUNS / "hp_heatmap.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Saved: {TRACKER_RUNS}/hp_heatmap.png")


## 5 — Annotated Frame Snapshots

12 evenly-spaced frames from the first test video for each tracker, displayed in a grid.

In [ ]:
import math

# Pick demo from the eval subset so pred files are guaranteed to exist
demo_video = eval_videos[0] if eval_videos else None

def annotated_grid(video_path, model_path, tracker_yaml, hp={}, conf=CONF, iou=IOU, n_frames=12):
    tmp_yaml = _make_tracker_yaml(tracker_yaml, hp) if hp else None
    tracker_arg = str(tmp_yaml) if tmp_yaml else tracker_yaml
    model = YOLO(str(model_path))
    cap   = cv2.VideoCapture(str(video_path))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    results_list = list(model.track(
        source=str(video_path), conf=conf, iou=iou,
        tracker=tracker_arg, persist=False, verbose=False, stream=True))
    if tmp_yaml and tmp_yaml.exists():
        tmp_yaml.unlink()
    indices = [int(i*(total-1)/(n_frames-1)) for i in range(n_frames)]
    out_frames = []
    cap = cv2.VideoCapture(str(video_path))
    for fi in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue
        r = results_list[min(fi, len(results_list)-1)]
        boxes, tids, cids, confs = [], [], [], []
        if r.boxes is not None and r.boxes.id is not None:
            for box in r.boxes:
                if box.id is None: continue
                boxes.append(box.xyxy[0].tolist())
                tids.append(int(box.id.item()))
                cids.append(int(box.cls.item()))
                confs.append(float(box.conf.item()))
        out_frames.append(draw_tracks(frame, boxes, tids, cids, CLASSES, confs))
    cap.release()
    return out_frames

if demo_video:
    for tracker in TRACKERS:
        tname = tracker_name(tracker)
        # Run both baseline and best-HP so we can visually compare
        _snap_configs = {
            "Baseline (arch)": final_results.get("baseline", {}).get("hp", _build_trial_hp({})),
            "Best HP":         final_results.get("best_hp",  {}).get("hp", best_configs.get(tracker, {})),
        }
        cols = 4
        for cfg_label, snap_hp in _snap_configs.items():
            snap_tracker_hp = {k: v for k, v in snap_hp.items() if k in _TRACKER_KEYS}
            snap_conf = snap_tracker_hp.pop("det_conf", CONF)
            snap_iou  = snap_tracker_hp.pop("det_iou",  IOU)
            frames = annotated_grid(demo_video, FINAL_WEIGHTS, tracker,
                                    hp=snap_tracker_hp, conf=snap_conf, iou=snap_iou, n_frames=12)
            rows_n = math.ceil(len(frames) / cols)
            fig, axes = plt.subplots(rows_n, cols, figsize=(cols*4, rows_n*3))
            axes = axes.flatten()
            for ax, fr in zip(axes, frames):
                ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis("off")
            for ax in axes[len(frames):]:
                ax.axis("off")
            _slug = cfg_label.lower().replace(' ', '_').replace('(', '').replace(')', '')
            plt.suptitle(f"{tname} [{cfg_label}] — annotated frames  ({demo_video.name})", fontsize=12)
            plt.tight_layout()
            out_path = TRACKER_RUNS / f"annotated_{tname}_{_slug}.png"
            plt.savefig(out_path, dpi=120, bbox_inches="tight")
            plt.show()
            print(f"Saved: {out_path}")
else:
    print("No test video found — skipping frame snapshots.")


## 6 — Trajectory Plot

Centroid path of every tracked object over the first test video for the best-MOTA
tracker, overlaid on the first frame.

In [ ]:
from helpers import _load_gt_trajectories, _draw_trajectory_ax

## 7 — Real-Time Interactive Tracking

Opens an OpenCV window and runs the best tracker live on a chosen video.
Press **`q`** to quit, **`space`** to pause/resume.

In [ ]:
# ── Configure ─────────────────────────────────────────────────────────────
RT_VIDEO   = demo_video          # or Path("path/to/video.mp4") or 0 for webcam
RT_TRACKER = best_tracker_yaml
RT_HP      = best_configs.get(RT_TRACKER, {})
# ──────────────────────────────────────────────────────────────────────────

if RT_VIDEO is None and not isinstance(RT_VIDEO, int):
    print("RT_VIDEO is None — skipping real-time cell.")
else:
    _tmp_yaml = _make_tracker_yaml(RT_TRACKER, RT_HP) if RT_HP else None
    _t_arg    = str(_tmp_yaml) if _tmp_yaml else RT_TRACKER
    _model    = YOLO(str(FINAL_WEIGHTS))
    _src      = str(RT_VIDEO) if not isinstance(RT_VIDEO, int) else RT_VIDEO

    _cap = cv2.VideoCapture(_src)
    if not _cap.isOpened():
        print(f"Cannot open: {RT_VIDEO}")
    else:
        print("Press 'q' to quit, 'space' to pause/resume.")
        _paused = False
        for _r in _model.track(source=_src, conf=CONF, iou=IOU,
                                tracker=_t_arg, persist=True, verbose=False, stream=True):
            _ret, _frame = _cap.read()
            if not _ret: break
            _boxes, _tids, _cids, _confs = [], [], [], []
            if _r.boxes is not None and _r.boxes.id is not None:
                for _b in _r.boxes:
                    if _b.id is None: continue
                    _boxes.append(_b.xyxy[0].tolist())
                    _tids.append(int(_b.id.item()))
                    _cids.append(int(_b.cls.item()))
                    _confs.append(float(_b.conf.item()))
            _vis = draw_tracks(_frame, _boxes, _tids, _cids, CLASSES, _confs)
            cv2.imshow("KARTector — real-time  [q=quit  space=pause]", _vis)
            key = cv2.waitKey(1) & 0xFF
            if key == ord("q"): break
            if key == ord(" "):
                _paused = not _paused
                while _paused:
                    k2 = cv2.waitKey(50) & 0xFF
                    if k2 in (ord(" "), ord("q")): _paused = False
        _cap.release()
        cv2.destroyAllWindows()
        if _tmp_yaml and _tmp_yaml.exists(): _tmp_yaml.unlink()
        print("Real-time session ended.")


## Save YOLO/BoTSORT Final Config

In [ ]:
# ── Save YOLO final config (BoTSORT) ─────────────────────────────────────────
import json as _jsave, yaml as _ysave

# Pull from final_results for the authoritative post-eval HP
_botsort_best = final_results.get("best_hp", {}).get("hp",
                    best_configs.get(list(best_configs.keys())[0] if best_configs else "botsort.yaml", {}))
_botsort_hp   = {k: v for k, v in _botsort_best.items() if k in _TRACKER_KEYS}

# Load fixed knobs from arch yaml and merge
_arch_yaml_path = REPO_ROOT / _arch['tracker_yaml']
_arch_yaml_vals = _ysave.safe_load(_arch_yaml_path.read_text()) if _arch_yaml_path.exists() else {}
_botsort_hp.setdefault('track_buffer', _arch_yaml_vals.get('track_buffer', 30))
_botsort_hp.setdefault('match_thresh', _arch_yaml_vals.get('match_thresh', 0.70))

# Write the final tuned tracker YAML
_final_yaml = REPO_ROOT / 'configs' / 'kartector_botsort_yolo_final.yaml'
_final_yaml_dict = {
    'tracker_type':      'botsort',
    'track_high_thresh': _botsort_hp.get('track_high_thresh', 0.25),
    'track_low_thresh':  _botsort_hp.get('track_low_thresh', 0.10),
    'new_track_thresh':  _botsort_hp.get('new_track_thresh', 0.30),
    'track_buffer':      int(_botsort_hp.get('track_buffer', 30)),
    'match_thresh':      float(_botsort_hp.get('match_thresh', 0.70)),
    'fuse_score':        True,
    'gmc_method':        'sparseOptFlow',
    'proximity_thresh':  float(_botsort_hp.get('proximity_thresh', 0.5)),
    'appearance_thresh': float(_botsort_hp.get('appearance_thresh', 0.85)),
    'with_reid':         bool(_arch_yaml_vals.get('with_reid', False)),
}
# If Option B (boxmot ReID) was used, override with_reid=True and note it
if REID_OPTION == 'B':
    _final_yaml_dict['with_reid'] = True
    # boxmot is used at runtime — the yaml is only for record-keeping
_final_yaml.write_text(_ysave.dump(_final_yaml_dict))

# Write the full workflow JSON (read by 05c and later notebooks)
_yolo_cfg = {
    'backbone':      'yolo',
    'weights':       str((YOLO_RUNS / 'final' / 'weights' / 'best.pt').relative_to(REPO_ROOT)),
    'tracker':       'botsort',
    'tracker_yaml':  str(_final_yaml.relative_to(REPO_ROOT)),
    'conf':          float(CONF),
    'iou':           float(IOU),
    'post_process':  bool(POST_PROCESS),
    'max_gap':       int(MAX_GAP),
    'max_dist_pct':  float(MAX_DIST_PCT),
    'reid_option':   REID_OPTION,       # 'A', 'B', or 'none'
    'reid_b_params': REID_B_PARAMS,     # boxmot params (used when reid_option=='B')
    'best_hp':       _botsort_hp,
}
_out = REPO_ROOT / 'configs' / 'kartector_botsort_yolo_final.json'
_out.write_text(_jsave.dumps(_yolo_cfg, indent=2))
print(f'YOLO/BoTSORT config saved: {_out}')
print(_jsave.dumps(_yolo_cfg, indent=2))


## RT-DETR Score Threshold Calibration

Re-runs the HP search with **RT-DETR** as the detector backbone.
Motion/association knobs are frozen to the best YOLO values;
only score-gating thresholds are swept.

In [ ]:
from ultralytics import RTDETR as _RTDETR

RTDETR_RUNS   = (REPO_ROOT / 'runs/rtdetr').resolve()
FINAL_RTDETR_WEIGHTS = RTDETR_RUNS / 'final' / 'weights' / 'best.pt'

if not FINAL_RTDETR_WEIGHTS.exists():
    print(f'WARNING: RT-DETR weights not found at {FINAL_RTDETR_WEIGHTS} — skipping RT-DETR section.')
    _rtdetr_available = False
else:
    _rtdetr_available = True

# Score-threshold grid only — motion knobs frozen from YOLO best config
RTDETR_HP_GRID = {
    'conf':             [0.10, 0.15, 0.20, 0.25, 0.30, 0.40],
    'track_high_thresh': [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60],
    'track_low_thresh':  [0.01, 0.03, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40],
    'new_track_thresh':  [0.15, 0.20, 0.25, 0.35, 0.45, 0.55, 0.65],
}
RTDETR_HP_MAX_TRIALS = HP_MAX_TRIALS   # reuse same budget

# Frozen motion knobs from YOLO best
# Pull from final_results for consistency with cells 20+
_yolo_best_hp = final_results.get("best_hp", {}).get("hp",
                    best_configs.get(list(best_configs.keys())[0] if best_configs else "botsort.yaml", {}))
_frozen_motion = {
    'track_buffer':      _yolo_best_hp.get('track_buffer',      _arch_yaml_vals.get('track_buffer', 30)),
    'match_thresh':      _yolo_best_hp.get('match_thresh',       _arch_yaml_vals.get('match_thresh', 0.70)),
    'proximity_thresh':  _yolo_best_hp.get('proximity_thresh',   0.5),
    'appearance_thresh': _yolo_best_hp.get('appearance_thresh',  0.85),
    'with_reid':         _arch_yaml_vals.get('with_reid', False),
}
print('Frozen motion knobs (from YOLO BoTSORT best):')
for k, v in _frozen_motion.items(): print(f'  {k}: {v}')

import tempfile, yaml as _yaml2
from ultralytics import YOLO as _YOLO2

def _run_tracker_rtdetr_botsort(weights_path, video_paths, output_dir, hp, pp_overrides=None):
    """RT-DETR detect + Ultralytics BoTSORT tracker via model.track()."""
    import yaml as _y
    # Build a temp botsort yaml merging frozen motion + score thresholds
    cfg = {
        'tracker_type':      'botsort',
        'track_high_thresh': hp.get('track_high_thresh', 0.25),
        'track_low_thresh':  hp.get('track_low_thresh', 0.10),
        'new_track_thresh':  hp.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_motion['track_buffer']),
        'match_thresh':      float(_frozen_motion['match_thresh']),
        'fuse_score':        True,
        'gmc_method':        'sparseOptFlow',
        'proximity_thresh':  float(_frozen_motion['proximity_thresh']),
        'appearance_thresh': float(_frozen_motion['appearance_thresh']),
        'with_reid':         bool(_frozen_motion['with_reid']),
    }
    if _frozen_motion.get('with_reid'):
        cfg['model'] = 'auto'
    tmp = Path(tempfile.mktemp(suffix='.yaml'))
    tmp.write_text(_y.dump(cfg))

    pp = pp_overrides or {}
    merge_time  = pp.get('pp_merge_time',  MAX_GAP      if POST_PROCESS else None)
    merge_dist  = pp.get('pp_merge_dist',  MAX_DIST_PCT if POST_PROCESS else 30.0)
    merge_interp= pp.get('pp_merge_interp', True)

    model = _RTDETR(str(weights_path))
    run_conf = hp.get('conf', CONF)
    output_dir = Path(output_dir); output_dir.mkdir(parents=True, exist_ok=True)
    all_preds = {}
    try:
        for vp in video_paths:
            vp = Path(vp)
            if not vp.exists(): print(f'  [SKIP] {vp.name}'); continue
            results = model.track(source=str(vp), conf=run_conf, iou=IOU,
                                  tracker=str(tmp), persist=False, verbose=False, stream=True)
            rows = []
            for fi, r in enumerate(results, 1):
                if r.boxes is None or r.boxes.id is None: continue
                for box in r.boxes:
                    if box.id is None: continue
                    x1, y1, x2, y2 = box.xyxy[0].tolist()
                    rows.append([fi, int(box.id.item()), x1, y1, x2-x1, y2-y1,
                                 float(box.conf.item()), int(box.cls.item())])
            rows = _merge_track_rows(rows, merge_time=merge_time,
                                     merge_dist=merge_dist, interpolate=merge_interp)
            (output_dir / f"{vp.stem}.txt").write_text("\n".join(
                f"{r[0]},{r[1]},{r[2]:.2f},{r[3]:.2f},{r[4]:.2f},{r[5]:.2f},{r[6]:.4f},{r[7]}"
                for r in rows))
            all_preds[vp.stem] = rows
            print(f'  {vp.name}: {len(rows)} detections')
    finally:
        if tmp.exists(): tmp.unlink()
    return all_preds

RTDETR_TRACKER_RUNS = (REPO_ROOT / 'runs/trackers_rtdetr_botsort').resolve()
RTDETR_TRACKER_RUNS.mkdir(parents=True, exist_ok=True)

rtdetr_results = {}
rtdetr_best_config = {}
rtdetr_hp_log = []

if _rtdetr_available:
    import itertools as _it2, random as _rand2
    _combos_r = list(_it2.product(*RTDETR_HP_GRID.values()))
    _r_keys   = list(RTDETR_HP_GRID.keys())
    _rand2.seed(42)
    _rand2.shuffle(_combos_r)
    _combos_r = [dict(zip(_r_keys, c)) for c in _combos_r[:RTDETR_HP_MAX_TRIALS]]
    print(f'\nRT-DETR sweep: {len(_combos_r)} trials')

    _rtdetr_hp_rows = []
    for idx, hp in enumerate(_combos_r):
        print(f'  Trial {idx+1}/{len(_combos_r)}: {hp}', flush=True)
        pred_dir_r = RTDETR_TRACKER_RUNS / 'botsort' / 'hp_search' / f'trial_{idx:03d}'
        try:
            _run_tracker_rtdetr_botsort(FINAL_RTDETR_WEIGHTS, search_videos, pred_dir_r, hp)
            m = compute_mot_metrics(MOT_DIR / "test", pred_dir_r)
            df_cnt_r = compute_count_metrics(MOT_DIR / "test", pred_dir_r, CLASSES)
            per_cls_r = df_cnt_r[df_cnt_r["class"] != "MACRO_AVG"]
            import warnings as _wrt
            with _wrt.catch_warnings():
                _wrt.simplefilter("ignore", RuntimeWarning)
                ace_r  = float(per_cls_r["ACE"].mean()) if not per_cls_r.empty else None
                bias_r = float(per_cls_r["CE"].mean())  if not per_cls_r.empty else None
            comp_r = ((ace_r + BIAS_PENALTY * abs(bias_r))
                      if (ace_r is not None and bias_r is not None) else None)
            row_r = {"trial": idx, **hp,
                     "MOTA": m.get("MOTA"), "IDF1": m.get("IDF1"), "ID_Sw": m.get("ID_Sw"),
                     "ACE": ace_r, "CE_bias": bias_r, "composite": comp_r}
            _rtdetr_hp_rows.append(row_r)
            print(f'    MOTA={m["MOTA"]}  IDF1={m["IDF1"]}  ACE={ace_r}  composite={comp_r}', flush=True)
        except Exception as _e_rt:
            print(f'  [ERROR trial {idx}] {_e_rt}', flush=True)
            continue

    df_rtdetr_hp = pd.DataFrame(_rtdetr_hp_rows)
    df_rtdetr_hp.to_csv(RTDETR_TRACKER_RUNS / 'botsort_hp_search_log.csv', index=False)

    # ── Pick best RT-DETR config by HP_SEARCH_METRIC ──────────────────────
    _METRIC_ASCENDING_R = {"MOTA": False, "IDF1": False, "ID_Sw": True,
                           "ACE": True, "composite": True, "CE_bias": True}
    _sort_col_r  = HP_SEARCH_METRIC
    _asc_r       = _METRIC_ASCENDING_R.get(_sort_col_r, False)
    _valid_r     = df_rtdetr_hp.dropna(subset=[_sort_col_r]) if not df_rtdetr_hp.empty else df_rtdetr_hp
    if not _valid_r.empty:
        _best_row_r   = _valid_r.sort_values(_sort_col_r, ascending=_asc_r).iloc[0]
        rtdetr_best_config = {k: _best_row_r[k].item() if hasattr(_best_row_r[k], 'item') else _best_row_r[k]
                               for k in RTDETR_HP_GRID if k in _best_row_r.index}
        print(f'\nBest RT-DETR BoTSORT config (by {_sort_col_r}):')
        print(rtdetr_best_config)
        print(df_rtdetr_hp.sort_values(_sort_col_r, ascending=_asc_r).head(10).to_string(index=False))
    else:
        print('No valid RT-DETR trials.')
else:
    print('Skipped — RT-DETR weights not found.')


## Save RT-DETR/BoTSORT Final Config

In [ ]:
# ── Save RT-DETR final config (BoTSORT) ──────────────────────────────────────
import json as _jsave2, yaml as _ysave2

if _rtdetr_available and rtdetr_best_config:
    _rtdetr_hp = dict(rtdetr_best_config)
    _rtdetr_conf = _rtdetr_hp.pop('conf', CONF)

    _rtdetr_yaml = REPO_ROOT / 'configs' / 'kartector_botsort_rtdetr_final.yaml'
    _rtdetr_yaml_dict = {
        'tracker_type':      'botsort',
        'track_high_thresh': _rtdetr_hp.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rtdetr_hp.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rtdetr_hp.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_motion['track_buffer']),
        'match_thresh':      float(_frozen_motion['match_thresh']),
        'fuse_score':        True,
        'gmc_method':        'sparseOptFlow',
        'proximity_thresh':  float(_frozen_motion['proximity_thresh']),
        'appearance_thresh': float(_frozen_motion['appearance_thresh']),
        'with_reid':         bool(_frozen_motion['with_reid']),
    }
    if _frozen_motion.get('with_reid'):
        _rtdetr_yaml_dict['model'] = 'auto'
    _rtdetr_yaml.write_text(_ysave2.dump(_rtdetr_yaml_dict))

    _rtdetr_cfg = {
        'backbone':      'rtdetr',
        'weights':       str((RTDETR_RUNS / 'final' / 'weights' / 'best.pt').relative_to(REPO_ROOT)),
        'tracker':       'botsort',
        'tracker_yaml':  str(_rtdetr_yaml.relative_to(REPO_ROOT)),
        'conf':          float(_rtdetr_conf),
        'iou':           float(IOU),
        'post_process':  bool(POST_PROCESS),
        'max_gap':       int(MAX_GAP),
        'max_dist_pct':  float(MAX_DIST_PCT),
        'best_hp':       rtdetr_best_config,
    }
    _out2 = REPO_ROOT / 'configs' / 'kartector_botsort_rtdetr_final.json'
    _out2.write_text(_jsave2.dumps(_rtdetr_cfg, indent=2))
    print(f'RT-DETR/BoTSORT config saved: {_out2}')
    print(_jsave2.dumps(_rtdetr_cfg, indent=2))
else:
    print('RT-DETR config not saved — run the sweep above first.')


## Annotated Frame Snapshots (RT-DETR)

12 evenly-spaced frames from the first eval video with RT-DETR+BoTSORT tracks overlaid.

In [ ]:
if _rtdetr_available and rtdetr_best_config and demo_video:
    from ultralytics import RTDETR as _RTDETR_VIZ

    _rt_viz_hp   = dict(rtdetr_best_config)
    _rt_viz_conf = _rt_viz_hp.pop('conf', CONF)
    _rt_viz_tkey = {k: v for k, v in _rt_viz_hp.items() if k in _TRACKER_KEYS}

    import tempfile as _tmpviz, yaml as _yamlviz
    _rt_viz_cfg = {
        'tracker_type':      'botsort',
        'track_high_thresh': _rt_viz_tkey.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rt_viz_tkey.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rt_viz_tkey.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_motion['track_buffer']),
        'match_thresh':      float(_frozen_motion['match_thresh']),
        'fuse_score':        True,
        'gmc_method':        'sparseOptFlow',
        'proximity_thresh':  float(_frozen_motion['proximity_thresh']),
        'appearance_thresh': float(_frozen_motion['appearance_thresh']),
        'with_reid':         bool(_frozen_motion['with_reid']),
    }
    if _frozen_motion.get('with_reid'):
        _rt_viz_cfg['model'] = 'auto'
    _rt_viz_tmp = Path(_tmpviz.mktemp(suffix='.yaml'))
    _rt_viz_tmp.write_text(_yamlviz.dump(_rt_viz_cfg))

    _rt_model_viz = _RTDETR_VIZ(str(FINAL_RTDETR_WEIGHTS))
    _rt_cap_viz   = cv2.VideoCapture(str(demo_video))
    _rt_total_viz = int(_rt_cap_viz.get(cv2.CAP_PROP_FRAME_COUNT)); _rt_cap_viz.release()
    _rt_results_viz = list(_rt_model_viz.track(
        source=str(demo_video), conf=_rt_viz_conf, iou=IOU,
        tracker=str(_rt_viz_tmp), persist=False, verbose=False, stream=True))
    if _rt_viz_tmp.exists(): _rt_viz_tmp.unlink()

    _rt_indices = [int(i * (_rt_total_viz - 1) / 11) for i in range(12)]
    _rt_frames_out = []
    _rt_cap2 = cv2.VideoCapture(str(demo_video))
    for _fi in _rt_indices:
        _rt_cap2.set(cv2.CAP_PROP_POS_FRAMES, _fi)
        _ret, _frame = _rt_cap2.read()
        if not _ret: continue
        _r = _rt_results_viz[min(_fi, len(_rt_results_viz) - 1)]
        _boxes, _tids, _cids, _confs = [], [], [], []
        if _r.boxes is not None and _r.boxes.id is not None:
            for _box in _r.boxes:
                if _box.id is None: continue
                _boxes.append(_box.xyxy[0].tolist())
                _tids.append(int(_box.id.item()))
                _cids.append(int(_box.cls.item()))
                _confs.append(float(_box.conf.item()))
        _rt_frames_out.append(draw_tracks(_frame, _boxes, _tids, _cids, CLASSES, _confs))
    _rt_cap2.release()

    import math as _math
    _cols = 4; _rows_n = _math.ceil(len(_rt_frames_out) / _cols)
    fig, axes = plt.subplots(_rows_n, _cols, figsize=(_cols * 4, _rows_n * 3))
    axes = axes.flatten()
    for ax, fr in zip(axes, _rt_frames_out):
        ax.imshow(cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)); ax.axis('off')
    for ax in axes[len(_rt_frames_out):]: ax.axis('off')
    plt.suptitle(f'RT-DETR+BoTSORT — annotated frames  ({Path(demo_video).name})', fontsize=12)
    plt.tight_layout()
    _out_snap_rt = RTDETR_TRACKER_RUNS / 'annotated_rtdetr_botsort.png'
    plt.savefig(_out_snap_rt, dpi=120, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out_snap_rt}')
else:
    print('Skipped RT-DETR snapshots — weights not found or sweep not run.')


## Trajectory Plot (RT-DETR)

Centroid paths for the best RT-DETR+BoTSORT config on the demo video.

In [ ]:
if _rtdetr_available and rtdetr_best_config and demo_video:
    _rt_traj_hp   = dict(rtdetr_best_config)
    _rt_traj_conf = _rt_traj_hp.pop('conf', CONF)
    _rt_traj_tkey = {k: v for k, v in _rt_traj_hp.items() if k in _TRACKER_KEYS}

    import tempfile as _tmptraj2, yaml as _yamltraj2
    _rt_traj_cfg = {
        'tracker_type':      'botsort',
        'track_high_thresh': _rt_traj_tkey.get('track_high_thresh', 0.25),
        'track_low_thresh':  _rt_traj_tkey.get('track_low_thresh', 0.10),
        'new_track_thresh':  _rt_traj_tkey.get('new_track_thresh', 0.30),
        'track_buffer':      int(_frozen_motion['track_buffer']),
        'match_thresh':      float(_frozen_motion['match_thresh']),
        'fuse_score':        True,
        'gmc_method':        'sparseOptFlow',
        'proximity_thresh':  float(_frozen_motion['proximity_thresh']),
        'appearance_thresh': float(_frozen_motion['appearance_thresh']),
        'with_reid':         bool(_frozen_motion['with_reid']),
    }
    if _frozen_motion.get('with_reid'):
        _rt_traj_cfg['model'] = 'auto'
    _rt_traj_tmp = Path(_tmptraj2.mktemp(suffix='.yaml'))
    _rt_traj_tmp.write_text(_yamltraj2.dump(_rt_traj_cfg))

    _rt_model_traj = _RTDETR_VIZ(str(FINAL_RTDETR_WEIGHTS))
    _rt_pred_traj = {}
    for _fi2, _r2 in enumerate(_rt_model_traj.track(
            source=str(demo_video), conf=_rt_traj_conf, iou=IOU,
            tracker=str(_rt_traj_tmp), persist=False, verbose=False, stream=True), 1):
        if _r2.boxes is None or _r2.boxes.id is None: continue
        for _box2 in _r2.boxes:
            if _box2.id is None: continue
            _tid2 = int(_box2.id.item()); _cid2 = int(_box2.cls.item())
            _x1, _y1, _x2, _y2 = _box2.xyxy[0].tolist()
            _rt_pred_traj.setdefault(_tid2, []).append(((_x1+_x2)/2, (_y1+_y2)/2, _fi2, _cid2))
    if _rt_traj_tmp.exists(): _rt_traj_tmp.unlink()

    _gt_traj_rt = _load_gt_trajectories(demo_video, MOT_DIR)
    _cap_bg_rt  = cv2.VideoCapture(str(demo_video)); _ret_bg, _bg_rt = _cap_bg_rt.read(); _cap_bg_rt.release()
    if not _ret_bg: _bg_rt = np.zeros((720, 1280, 3), dtype=np.uint8)

    _ncols_rt = 2 if _gt_traj_rt else 1
    fig, axes = plt.subplots(1, _ncols_rt, figsize=(12 * _ncols_rt, 7))
    if _ncols_rt == 1: axes = [axes]
    _draw_trajectory_ax(axes[0], _rt_pred_traj,
                        f'Predicted (RT-DETR+BoTSORT)  ({Path(demo_video).name})', _bg_rt)
    if _gt_traj_rt:
        _draw_trajectory_ax(axes[1], _gt_traj_rt,
                            f'Ground Truth  ({Path(demo_video).name})', _bg_rt)
    plt.tight_layout()
    _out_traj_rt = RTDETR_TRACKER_RUNS / 'trajectories_rtdetr.png'
    plt.savefig(_out_traj_rt, dpi=150, bbox_inches='tight'); plt.show()
    print(f'Saved: {_out_traj_rt}')
else:
    print('Skipped RT-DETR trajectory plot — weights not found or sweep not run.')

